In [1]:
import pickle
import numpy as np
import random
import os
from datetime import datetime

DB_FILE = "bank_data.pkl"

def load_data():
    """File se account data load karta hai (pickle use karke)"""
    if os.path.exists(DB_FILE):
        try:
            with open(DB_FILE, 'rb') as f:
                return pickle.load(f)
        except Exception:
            return {}
    return {}

def save_data(accounts):
    """File mein account data save karta hai"""
    with open(DB_FILE, 'wb') as f:
        pickle.dump(accounts, f)

# Lambda helper functions (Bonus Requirement)
is_positive = lambda amt: amt > 0
generate_acc_num = lambda existing: str(random.randint(10000, 99999)) if not existing else generate_acc_num(existing)

def create_account(accounts):
    """Naya account create karne ke liye"""
    print("\n--- CREATE NEW ACCOUNT ---")
    name = input("Account Holder Ka Naam Likhein: ").strip()
    if not name:
        print("Error: Naam khali nahi ho sakta!")
        return

    acc_type = input("Account Type (Savings/Current): ").strip().capitalize()
    if acc_type not in ['Savings', 'Current']:
        print("Error: Invalid account type! Sirf Savings ya Current choose karein.")
        return

    try:
        initial_balance = float(input("Initial Deposit Amount (Min 500): "))
        if initial_balance < 0:
            print("Error: Deposit amount negative nahi ho sakta!")
            return
    except ValueError:
        print("Error: Valid number enter karein!")
        return

    password = input("Account Password Set Karein (Bonus Feature): ").strip()

    # Auto-generate 5 digit unique account number
    acc_num = str(random.randint(10000, 99999))
    while acc_num in accounts:
        acc_num = str(random.randint(10000, 99999))

    accounts[acc_num] = {
        'name': name,
        'type': acc_type,
        'balance': initial_balance,
        'password': password,
        'transactions': []
    }

    # Initial transaction history log
    if initial_balance > 0:
        accounts[acc_num]['transactions'].append({
            'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'type': 'Deposit',
            'amount': initial_balance
        })

    save_data(accounts)
    print(f"\n Success: Account Successfully Ban Gaya!")
    print(f" Aapka Account Number Hai: {acc_num}")

def authenticate_user(accounts):
    """User Login Authentication Check"""
    acc_num = input("Account Number Enter Karein: ").strip()
    if acc_num not in accounts:
        print("Error: Account nahi mila!")
        return None

    pwd = input("Password Enter Karein: ").strip()
    if accounts[acc_num]['password'] == pwd:
        return acc_num
    else:
        print("Error: Galat Password!")
        return None

def view_account_details(accounts):
    """Account Details View Karna"""
    print("\n--- VIEW ACCOUNT DETAILS ---")
    acc_num = authenticate_user(accounts)
    if acc_num:
        acc = accounts[acc_num]
        print("\n=== Account Details ===")
        print(f"Account Number: {acc_num}")
        print(f"Account Holder: {acc['name']}")
        print(f"Account Type  : {acc['type']}")
        print(f"Current Balance: Rs. {acc['balance']:.2f}")

def deposit_money(accounts):
    """Account Mein Paise Jama Karna"""
    print("\n--- DEPOSIT MONEY ---")
    acc_num = authenticate_user(accounts)
    if acc_num:
        try:
            amount = float(input("Deposit Amount Enter Karein: "))
            if not is_positive(amount):
                print("Error: Amount positive hona chahiye!")
                return

            accounts[acc_num]['balance'] += amount
            accounts[acc_num]['transactions'].append({
                'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'type': 'Deposit',
                'amount': amount
            })
            save_data(accounts)
            print(f" Success: Rs. {amount} Deposit Ho Gaye! Naya Balance: Rs. {accounts[acc_num]['balance']:.2f}")
        except ValueError:
            print("Error: Valid Amount Enter Karein!")

def withdraw_money(accounts):
    """Account Se Paise Nikalna"""
    print("\n--- WITHDRAW MONEY ---")
    acc_num = authenticate_user(accounts)
    if acc_num:
        try:
            amount = float(input("Withdrawal Amount Enter Karein: "))
            if not is_positive(amount):
                print("Error: Amount positive hona chahiye!")
                return

            if amount > accounts[acc_num]['balance']:
                print("Error: Insufficient Balance! Account me utne paise nahi hain.")
                return

            accounts[acc_num]['balance'] -= amount
            accounts[acc_num]['transactions'].append({
                'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'type': 'Withdrawal',
                'amount': amount
            })
            save_data(accounts)
            print(f" Success: Rs. {amount} Withdraw Ho Gaye! Naya Balance: Rs. {accounts[acc_num]['balance']:.2f}")
        except ValueError:
            print("Error: Valid Amount Enter Karein!")

def transfer_money(accounts):
    """Ek Account Se Dusre Account Mein Money Transfer"""
    print("\n--- TRANSFER FUNDS ---")
    sender_acc = authenticate_user(accounts)
    if sender_acc:
        target_acc = input("Receiver Ka Account Number Enter Karein: ").strip()
        if target_acc not in accounts:
            print("Error: Receiver account nahi mila!")
            return
        if target_acc == sender_acc:
            print("Error: Apne hi account me transfer nahi kar sakte!")
            return

        try:
            amount = float(input("Transfer Amount Enter Karein: "))
            if not is_positive(amount):
                print("Error: Amount positive hona chahiye!")
                return

            if amount > accounts[sender_acc]['balance']:
                print("Error: Insufficient Balance!")
                return

            # Debit Sender
            accounts[sender_acc]['balance'] -= amount
            accounts[sender_acc]['transactions'].append({
                'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'type': f'Transfer Out to {target_acc}',
                'amount': amount
            })

            # Credit Receiver
            accounts[target_acc]['balance'] += amount
            accounts[target_acc]['transactions'].append({
                'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'type': f'Transfer In from {sender_acc}',
                'amount': amount
            })

            save_data(accounts)
            print(f" Success: Rs. {amount} Account {target_acc} me transfer ho gaye!")
        except ValueError:
            print("Error: Valid Amount Enter Karein!")

def view_transaction_history_and_stats(accounts):
    """Transaction History View Karna aur NumPy Statistics Calculate Karna"""
    print("\n--- TRANSACTION HISTORY & SUMMARY REPORT ---")
    acc_num = authenticate_user(accounts)
    if acc_num:
        txns = accounts[acc_num]['transactions']
        if not txns:
            print("Koi transaction record nahi mila.")
            return

        print("\n--- Transaction History ---")
        print(f"{'Date & Time':<22} | {'Type':<25} | {'Amount':<10}")
        print("-" * 63)

        deposits = []
        withdrawals = []
        all_amounts = []

        for t in txns:
            print(f"{t['date']:<22} | {t['type']:<25} | Rs.{t['amount']:<10.2f}")
            all_amounts.append(t['amount'])
            if 'Deposit' in t['type'] or 'Transfer In' in t['type']:
                deposits.append(t['amount'])
            elif 'Withdrawal' in t['type'] or 'Transfer Out' in t['type']:
                withdrawals.append(t['amount'])

        # NumPy Summary Statistics (Requirement #4)
        np_all = np.array(all_amounts) if all_amounts else np.array([0])
        np_dep = np.array(deposits) if deposits else np.array([0])
        np_wth = np.array(withdrawals) if withdrawals else np.array([0])

        print("\n=== NumPy Summary Statistics ===")
        print(f"Total Deposits Amount    : Rs. {np.sum(np_dep):.2f}")
        print(f"Total Withdrawals Amount : Rs. {np.sum(np_wth):.2f}")
        print(f"Average Transaction Amt  : Rs. {np.mean(np_all):.2f}")

def main_menu():
    """Interactive Command Line Menu Interface"""
    accounts = load_data()

    while True:
        print("\n==========================================")
        print("  PYTHON BANK ACCOUNT MANAGEMENT SYSTEM   ")
        print("==========================================")
        print("1. Open New Account")
        print("2. View Account Details")
        print("3. Deposit Money")
        print("4. Withdraw Money")
        print("5. Transfer Money")
        print("6. View Transaction History & NumPy Stats")
        print("7. Exit Program")

        choice = input("Option select karein (1-7): ").strip()

        if choice == '1':
            create_account(accounts)
        elif choice == '2':
            view_account_details(accounts)
        elif choice == '3':
            deposit_money(accounts)
        elif choice == '4':
            withdraw_money(accounts)
        elif choice == '5':
            transfer_money(accounts)
        elif choice == '6':
            view_transaction_history_and_stats(accounts)
        elif choice == '7':
            save_data(accounts)
            print("\nDhanyawad! Bank Management System Close Ho Raha Hai.")
            break
        else:
            print("Invalid Choice! Kripya 1 se 7 tak number enter karein.")

# Program Run Karein
if __name__ == "__main__":
    main_menu()


  PYTHON BANK ACCOUNT MANAGEMENT SYSTEM   
1. Open New Account
2. View Account Details
3. Deposit Money
4. Withdraw Money
5. Transfer Money
6. View Transaction History & NumPy Stats
7. Exit Program
Option select karein (1-7): Aditya Yadav
Invalid Choice! Kripya 1 se 7 tak number enter karein.

  PYTHON BANK ACCOUNT MANAGEMENT SYSTEM   
1. Open New Account
2. View Account Details
3. Deposit Money
4. Withdraw Money
5. Transfer Money
6. View Transaction History & NumPy Stats
7. Exit Program
Option select karein (1-7): 1

--- CREATE NEW ACCOUNT ---
Account Holder Ka Naam Likhein: Aditya Yadav
Account Type (Savings/Current): Savings
Initial Deposit Amount (Min 500): 5000
Account Password Set Karein (Bonus Feature): 1234

 Success: Account Successfully Ban Gaya!
 Aapka Account Number Hai: 53723

  PYTHON BANK ACCOUNT MANAGEMENT SYSTEM   
1. Open New Account
2. View Account Details
3. Deposit Money
4. Withdraw Money
5. Transfer Money
6. View Transaction History & NumPy Stats
7. Exit Program
